In [25]:
import pandas as pd
import numpy as np
import ast

In [26]:
oscar_df = pd.read_csv('../data/raw/the_oscar_award.csv')
tmdb_movies = pd.read_csv('../data/raw/tmdb_5000_movies.csv')

In [27]:
oscar_df.columns

Index(['year_film', 'year_ceremony', 'ceremony', 'category', 'canon_category',
       'name', 'film', 'winner'],
      dtype='str')

In [28]:
tmdb_movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='str')

In [29]:
oscar_df['film_lower'] = oscar_df['film'].str.lower()
tmdb_movies['title_lower'] = tmdb_movies['title'].str.lower()

In [30]:
oscar_df[['film', 'film_lower']].head()

,film,film_lower
0,The Noose,the noose
1,The Patent Leather Kid,the patent leather kid
2,The Last Command,the last command
3,The Way of All Flesh,the way of all flesh
4,A Ship Comes In,a ship comes in


In [31]:
tmdb_movies[['title', 'title_lower']].head()

,title,title_lower
0,Avatar,avatar
1,Pirates of the Caribbean: At World's End,pirates of the caribbean: at world's end
2,Spectre,spectre
3,The Dark Knight Rises,the dark knight rises
4,John Carter,john carter


In [32]:
oscar_winners = oscar_df.groupby('film_lower')['winner'].max().reset_index()
oscar_winners.rename(columns={'winner': 'is_oscar_winner'}, inplace=True)
oscar_winners['is_oscar_winner'] = oscar_winners['is_oscar_winner'].astype(int)

In [33]:
oscar_winners.head()

,film_lower,is_oscar_winner
0,"$1,000 a minute",0
1,'38',0
2,'crocodile' dundee,0
3,'round midnight,1
4,(a) torzija [(a) torsion],0


In [34]:
oscar_winners.info()

<class 'pandas.DataFrame'>
RangeIndex: 5116 entries, 0 to 5115
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   film_lower       5116 non-null   str  
 1   is_oscar_winner  5116 non-null   int64
dtypes: int64(1), str(1)
memory usage: 80.1 KB


In [35]:
df = pd.merge(tmdb_movies, oscar_winners, left_on='title_lower', right_on='film_lower', how='inner')

In [36]:
df.drop(columns=['film_lower', 'title_lower'], inplace=True)

In [37]:
df.shape[0]  # Movies after merge

985

In [43]:
df['is_oscar_winner'].sum().tolist()  # Oscar winners

387

In [39]:
df.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,is_oscar_winner
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,1
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,0
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,1


In [46]:
def extract_names(column_data):
    try:
        items = ast.literal_eval(column_data)  # Python list
        return [item['name'] for item in items]
    except (ValueError, SyntaxError, TypeError):
        return []

In [47]:
df['genres_list'] = df['genres'].apply(extract_names)

In [48]:
genres_dummies = df['genres_list'].str.join('|').str.get_dummies().add_prefix('genre_')
df = pd.concat([df, genres_dummies], axis=1)

In [49]:
df['keywords_list'] = df['keywords'].apply(extract_names)

In [50]:
keywords_dummies = df['keywords_list'].str.join('|').str.get_dummies().add_prefix('key_')
df = pd.concat([df, keywords_dummies], axis=1)

In [51]:
columns_to_drop = ['genres', 'genres_list', 'keywords', 'keywords_list', 'homepage', 'tagline', 'overview',
                   'original_title']

In [52]:
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

In [55]:
df.columns

Index(['budget', 'id', 'original_language', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages',
       ...
       'key_yuppie', 'key_zeppelin', 'key_zero gravity', 'key_zoo',
       'key_zookeeper', 'key_ begins with text', 'key_ flipping coin',
       'key_ gilbert and sullivan', 'key_ nightgown', 'key_ nosferatu'],
      dtype='str', length=4719)

In [56]:
df.columns = (
    df.columns
    .str.strip()  # Remove leading and trailing whitespaces
    .str.lower()  # Convert all letters to lowercase
    .str.replace(' ', '_')  # Replace all letters to lowercase
    .str.replace(r'[^a-z0-9_]', '', regex=True)  # Remove all special characters
    .str.replace(r'_+', '_', regex=True)  # Replace multiple consecutive underscores with a single one
)

In [58]:
df.columns[-20:].tolist()  # Random 20 columns

['key_yakuza',
 'key_yankee_stadium_bronx_new_york_city',
 'key_yokai',
 'key_young_adult',
 'key_young_boy',
 'key_young_entrepreneur',
 'key_young_love',
 'key_youngster',
 'key_youth',
 'key_yucatec_maya_language',
 'key_yuppie',
 'key_zeppelin',
 'key_zero_gravity',
 'key_zoo',
 'key_zookeeper',
 'key_begins_with_text',
 'key_flipping_coin',
 'key_gilbert_and_sullivan',
 'key_nightgown',
 'key_nosferatu']

In [59]:
df.columns

Index(['budget', 'id', 'original_language', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages',
       ...
       'key_yuppie', 'key_zeppelin', 'key_zero_gravity', 'key_zoo',
       'key_zookeeper', 'key_begins_with_text', 'key_flipping_coin',
       'key_gilbert_and_sullivan', 'key_nightgown', 'key_nosferatu'],
      dtype='str', length=4719)

In [61]:
df.to_csv('../data/processed/oscar_tmdb_cleaned.csv', index=False)